In [1]:
!pip install -q boto3

In [3]:
import os
import boto3


# ============================================================
# CONFIGURATION
# ============================================================

MODEL_DIR = "/home/jovyan/Iris_KubeFlow/iris_model/1"

MINIO_ENDPOINT = "http://minio-service.kubeflow.svc.cluster.local:9000"

MINIO_ACCESS_KEY = "minio"
MINIO_SECRET_KEY = "minio123"

BUCKET_NAME = "models"

MODEL_PREFIX = "iris/1"


# ============================================================
# 1. CHECK LOCAL MODEL
# ============================================================

print("=" * 70)
print("[1] Checking local SavedModel")
print("=" * 70)

print("Model directory:")
print(MODEL_DIR)

if not os.path.isdir(MODEL_DIR):
    raise RuntimeError(
        f"Model directory does not exist: {MODEL_DIR}"
    )

saved_model_pb = os.path.join(
    MODEL_DIR,
    "saved_model.pb"
)

variables_dir = os.path.join(
    MODEL_DIR,
    "variables"
)

if not os.path.isfile(saved_model_pb):
    raise RuntimeError(
        "saved_model.pb not found!"
    )

if not os.path.isdir(variables_dir):
    raise RuntimeError(
        "variables directory not found!"
    )

print("saved_model.pb: OK")
print("variables/: OK")


# ============================================================
# 2. SHOW LOCAL MODEL FILES
# ============================================================

print("\n" + "=" * 70)
print("[2] Local model files")
print("=" * 70)

for root, dirs, files in os.walk(MODEL_DIR):

    for file in files:

        file_path = os.path.join(
            root,
            file
        )

        size = os.path.getsize(
            file_path
        )

        print(
            file_path,
            "->",
            size,
            "bytes"
        )


# ============================================================
# 3. CONNECT TO MINIO
# ============================================================

print("\n" + "=" * 70)
print("[3] Connecting to MinIO")
print("=" * 70)

print(
    "Endpoint:",
    MINIO_ENDPOINT
)

s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    region_name="us-east-1"
)

print("MinIO client created")


# ============================================================
# 4. TEST MINIO CONNECTION
# ============================================================

print("\n" + "=" * 70)
print("[4] Testing MinIO connection")
print("=" * 70)

try:

    response = s3.list_buckets()

    print("MinIO connection successful!")

    print("\nExisting buckets:")

    for bucket in response.get(
        "Buckets",
        []
    ):

        print(
            "-",
            bucket["Name"]
        )

except Exception as e:

    print("\nMinIO connection FAILED")

    print(
        type(e).__name__,
        ":",
        str(e)
    )

    raise


# ============================================================
# 5. CHECK / CREATE BUCKET
# ============================================================

print("\n" + "=" * 70)
print("[5] Checking bucket")
print("=" * 70)

try:

    s3.head_bucket(
        Bucket=BUCKET_NAME
    )

    print(
        f"Bucket '{BUCKET_NAME}' already exists"
    )

except Exception:

    print(
        f"Bucket '{BUCKET_NAME}' does not exist"
    )

    print("Creating bucket...")

    s3.create_bucket(
        Bucket=BUCKET_NAME
    )

    print("Bucket created successfully")


# ============================================================
# 6. UPLOAD MODEL
# ============================================================

print("\n" + "=" * 70)
print("[6] Uploading model to MinIO")
print("=" * 70)

uploaded_count = 0

for root, dirs, files in os.walk(
    MODEL_DIR
):

    for file in files:

        local_file = os.path.join(
            root,
            file
        )

        relative_path = os.path.relpath(
            local_file,
            MODEL_DIR
        )

        # Windows/Linux path compatibility
        relative_path = relative_path.replace(
            "\\",
            "/"
        )

        minio_object = (
            MODEL_PREFIX
            + "/"
            + relative_path
        )

        print("\nUploading:")
        print("Local :", local_file)
        print("MinIO :", minio_object)

        s3.upload_file(
            local_file,
            BUCKET_NAME,
            minio_object
        )

        print("Status: SUCCESS")

        uploaded_count += 1


# ============================================================
# 7. VERIFY UPLOAD
# ============================================================

print("\n" + "=" * 70)
print("[7] Verifying uploaded model")
print("=" * 70)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=MODEL_PREFIX
)

objects = response.get(
    "Contents",
    []
)

print(
    "Objects found:",
    len(objects)
)

for obj in objects:

    print(
        obj["Key"],
        "->",
        obj["Size"],
        "bytes"
    )


# ============================================================
# 8. FINAL VALIDATION
# ============================================================

if len(objects) == 0:

    raise RuntimeError(
        "Model upload failed: "
        "no objects found in MinIO!"
    )

print("\n" + "=" * 70)
print("MODEL UPLOADED SUCCESSFULLY")
print("=" * 70)

print("\nUploaded files:", uploaded_count)

print("\nMinIO location:")

print(
    f"s3://{BUCKET_NAME}/{MODEL_PREFIX}"
)

print("\nKServe storageUri will be:")

print(
    f"s3://{BUCKET_NAME}/{MODEL_PREFIX}"
)

[1] Checking local SavedModel
Model directory:
/home/jovyan/Iris_KubeFlow/iris_model/1
saved_model.pb: OK
variables/: OK

[2] Local model files
/home/jovyan/Iris_KubeFlow/iris_model/1/saved_model.pb -> 53469 bytes
/home/jovyan/Iris_KubeFlow/iris_model/1/fingerprint.pb -> 58 bytes
/home/jovyan/Iris_KubeFlow/iris_model/1/variables/variables.index -> 940 bytes
/home/jovyan/Iris_KubeFlow/iris_model/1/variables/variables.data-00000-of-00001 -> 3861 bytes

[3] Connecting to MinIO
Endpoint: http://minio-service.kubeflow.svc.cluster.local:9000
MinIO client created

[4] Testing MinIO connection
MinIO connection successful!

Existing buckets:
- irisdataset
- mlpipeline

[5] Checking bucket
Bucket 'models' does not exist
Creating bucket...
Bucket created successfully

[6] Uploading model to MinIO

Uploading:
Local : /home/jovyan/Iris_KubeFlow/iris_model/1/saved_model.pb
MinIO : iris/1/saved_model.pb
Status: SUCCESS

Uploading:
Local : /home/jovyan/Iris_KubeFlow/iris_model/1/fingerprint.pb
MinIO :